In [ ]:
import os
import re
import pandas as pd
import numpy as np
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# for code run locally
# login to hugging face

splits = {
    "train": "task_a/task_a_training_set_1.parquet",
    "validation": "task_a/task_a_validation_set.parquet",
    "test": "task_a/task_a_test_set_sample.parquet",
}

base_path = "hf://datasets/DaniilOr/SemEval-2026-Task13/"

# Load each split into Pandas
train_data = pd.read_parquet(base_path + splits["train"])
val_data = pd.read_parquet(base_path + splits["validation"])
test_data = pd.read_parquet(base_path + splits["test"])

In [ ]:
# # run code on kaggle

# train_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_training_set_1.parquet"
# val_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_validation_set.parquet"
# test_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"


# # load train,test & validation data
# train_data = pd.read_parquet(train_path)
# val_data = pd.read_parquet(val_path)
# test_data = pd.read_parquet(test_path)

In [3]:
train_data.head()

,code,generator,label,language
0,"(a, b, c, d) = [int(x) for x in input().split(...",human,0,Python
1,valid version for the language; all others can...,Qwen/Qwen2.5-Coder-1.5B,1,Python
2,python\ndef min_cards_to_flip(s):\n vowels ...,Qwen/Qwen2.5-Coder-7B-Instruct,1,Python
3,T = int(input())\nfor t in range(T):\n\tcolor ...,human,0,Python
4,for i in range(int(input())):\n\tinput()\n\ta ...,human,0,Python


In [ ]:
# Basic cleaning
# ====================================================
#  Extra features
# ====================================================

camel_case_pattern = r"\b[a-z]+(?:[A-Z][a-z0-9]+)+\b"
snake_case_pattern = r"\b[a-z]+(?:_[a-z0-9]+)+\b"


def count_camel(text):
    return len(re.findall(camel_case_pattern, text))


def count_snake(text):
    return len(re.findall(snake_case_pattern, text))


# training dataset

train_data["code"] = train_data["code"].astype(str)

train_data["camel_count"] = train_data["code"].apply(count_camel)
train_data["snake_count"] = train_data["code"].apply(count_snake)

train_data["total_idents"] = train_data["camel_count"] + train_data["snake_count"]
train_data["camel_ratio"] = train_data["camel_count"] / (
    train_data["total_idents"] + 1e-5
)
train_data["snake_ratio"] = train_data["snake_count"] / (
    train_data["total_idents"] + 1e-5
)


# Code Features Extraction
train_data["code_length"] = train_data["code"].str.len()
train_data["num_comments"] = train_data["code"].str.count("#|//")
train_data["num_tabs"] = train_data["code"].str.count("\t")
train_data["num_spaces"] = train_data["code"].str.count("    ")

train_data["num_lines"] = train_data["code"].apply(lambda x: x.count("\n") + 1)
train_data["num_defs"] = train_data["code"].str.count(r"\bdef\b")
train_data["num_for"] = train_data["code"].str.count(r"\bfor\b")
train_data["num_if"] = train_data["code"].str.count(r"\bif\b")
train_data["num_import"] = train_data["code"].str.count(r"\bimport\b")

In [ ]:
# test dataset

test_data["code"] = test_data["code"].astype(str)

test_data["camel_count"] = test_data["code"].apply(count_camel)
test_data["snake_count"] = test_data["code"].apply(count_snake)
test_data["total_idents"] = test_data["camel_count"] + test_data["snake_count"]
test_data["camel_ratio"] = test_data["camel_count"] / (test_data["total_idents"] + 1e-5)
test_data["snake_ratio"] = test_data["snake_count"] / (test_data["total_idents"] + 1e-5)

# Code Features Extraction
test_data["code_length"] = test_data["code"].str.len()
test_data["num_comments"] = test_data["code"].str.count("#|//")
test_data["num_tabs"] = test_data["code"].str.count("\t")
test_data["num_spaces"] = test_data["code"].str.count("    ")

test_data["num_lines"] = test_data["code"].apply(lambda x: x.count("\n") + 1)
test_data["num_defs"] = test_data["code"].str.count(r"\bdef\b")
test_data["num_for"] = test_data["code"].str.count(r"\bfor\b")
test_data["num_if"] = test_data["code"].str.count(r"\bif\b")
test_data["num_import"] = test_data["code"].str.count(r"\bimport\b")

In [ ]:
# ====================================================
# Train-test split
# ====================================================

X_train_text, y_train = train_data["code"], train_data["label"]
X_test_text, y_test = test_data["code"], test_data["label"]
X_val_text, y_val = val_data["code"], val_data["label"]

## Hyperparameter Tuning Optuna Objective Function


In [ ]:
# ------------------------------------------------------
# Optuna Objective Function
# ------------------------------------------------------


# you can check it for any model by change the hyperparameters
def objective(trial):

    # TF-IDF hyperparameters
    max_features = trial.suggest_categorical("max_features", [10000, 20000, 30000])
    ngram_range = trial.suggest_categorical("ngram_range", [])
    min_df = trial.suggest_float("min_df", 1e-5, 0.01, log=True)

    # Logistic Regression hyperparameters
    C = trial.suggest_float("C", 1e-3, 10, log=True)
    solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    # Pipeline definition
    pipe = Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    strip_accents="unicode",
                    analyzer="word",
                    max_features=max_features,
                    ngram_range=ngram_range,
                    min_df=min_df,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    C=C,
                    solver=solver,
                    class_weight=class_weight,
                    max_iter=5000,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    # # Fit model
    pipe.fit(X_train_text, y_train)
    preds = pipe.predict(X_test_text)
    score = f1_score(y_test, preds, average="macro")

    # Compute cross-validated F1 score (macro average)
    # score = cross_val_score(pipe, X_train_text, y_train, cv=5, scoring="f1_macro").mean()

    return score


# ------------------------------------------------------
# Run Optuna Study
# ------------------------------------------------------
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # increase n_trials for better results

# Best hyperparameters
print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

# best_params = {'max_features': 30000, 'ngram_range': (1, 1), 'min_df': 0.0015430189491796615, 'C': 8.853443301355558, 'solver': 'liblinear', 'class_weight': 'balanced'}

In [ ]:
best_params = {
    "max_features": 30000,
    "ngram_range": (1, 1),
    "min_df": 0.0015430189491796615,
    "C": 8.853443301355558,
    "solver": "liblinear",
    "class_weight": "balanced",
}

In [ ]:
tfidf = TfidfVectorizer(
    strip_accents="unicode",
    analyzer="word",
    max_features=best_params["max_features"],
    ngram_range=best_params["ngram_range"],
    min_df=best_params["min_df"],
)


X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)
# validation data
X_val_tfidf = tfidf.transform(X_val_text)

In [ ]:
# ====================================================
#  Metadata features
# ====================================================

meta_features = [
    # 'language',
    "camel_count",
    "snake_count",
    "camel_ratio",
    "snake_ratio",
    "code_length",
    "num_comments",
    "num_tabs",
    "num_spaces",
    "num_lines",
    "num_defs",
    "num_for",
    "num_if",
    "num_import",
]

X_train_meta = train_data[meta_features].values
X_test_meta = test_data[meta_features].values


# Combine sparse and dense features
from scipy.sparse import hstack

X_train_meta_tfidf_data = hstack([X_train_tfidf, X_train_meta])
X_test_meta_tfidf_data = hstack([X_test_tfidf, X_test_meta])

In [ ]:
# ====================================================
#  Define Models
# ====================================================

models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Multinomial NB": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
}

# Optional (if installed)
try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(eval_metric="logloss", tree_method="hist")
except:
    pass

try:
    from lightgbm import LGBMClassifier

    models["LightGBM"] = LGBMClassifier()
except:
    pass

# ====================================================
# Train and Evaluate All Models
# ====================================================

results = []

for name, model in models.items():
    print("=" * 60)
    print(f"Training {name}...")

    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    print(classification_report(y_test, pred))

    results.append([name, acc, f1])

# Results summary
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1 Score"])
results_df = results_df.sort_values("F1 Score", ascending=False)

print("\n\n==== FINAL RESULTS ====")
print(results_df)

<!-- Train The Model on Combine  Metadata features and tfidf vectors -->


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Multinomial NB": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    # "Decision Tree": DecisionTreeClassifier(),
    # "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
}

# Optional (if installed)
try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(eval_metric="logloss", tree_method="hist")
except:
    pass


results = []

for name, model in models.items():
    print("=" * 60)
    print(f"Training {name}...")

    model.fit(X_train_meta_tfidf_data, y_train)
    pred = model.predict(X_test_meta_tfidf_data)

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    print(classification_report(y_test, pred))

    results.append([name, acc, f1])

# Results summary
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1 Score"])
results_df = results_df.sort_values("F1 Score", ascending=False)

print("\n\n==== FINAL RESULTS ====")
print(results_df)

### Best Model From Experiments Is LogisticRegression with TF-IDF


In [7]:
best_params

{'max_features': 30000,
 'ngram_range': (1, 1),
 'min_df': 0.0015430189491796615,
 'C': 8.853443301355558,
 'solver': 'liblinear',
 'class_weight': 'balanced'}

In [ ]:
logisticRegressionclf = LogisticRegression(
    C=best_params["C"],
    solver=best_params["solver"],
    class_weight=best_params["class_weight"],
    max_iter=5000,
    n_jobs=-1,
)

print("=" * 60)
logisticRegressionclf.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,8.853443301355558
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'liblinear'
,max_iter,5000
,multi_class,'deprecated'


In [ ]:
def custom_predict(model, X_data, threshold):
    """
    Generates class predictions based on a custom threshold.
    """
    probs = model.predict_proba(X_data)
    # Return 1 if the positive class probability is >= threshold, else 0
    return (probs[:, 1] >= threshold).astype(int)


# Usage:
pred = custom_predict(logisticRegressionclf, X_test_tfidf, threshold=0.6)


acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)

print(classification_report(y_test, pred))
print("=" * 60)
print(f"Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")

              precision    recall  f1-score   support

           0       0.90      0.41      0.57       777
           1       0.29      0.83      0.43       223

    accuracy                           0.51      1000
   macro avg       0.59      0.62      0.50      1000
weighted avg       0.76      0.51      0.54      1000

Accuracy: 0.5080, F1 Score: 0.4306


## feature importance


In [19]:
feature_names = tfidf.get_feature_names_out()
coefs = logisticRegressionclf.coef_[0]

In [20]:
top_k = 30
top_machine_idx = np.argsort(coefs)[-top_k:]
top_machine_features = [(feature_names[i], coefs[i]) for i in reversed(top_machine_idx)]

In [21]:
top_human_idx = np.argsort(coefs)[:top_k]
top_human_features = [(feature_names[i], coefs[i]) for i in top_human_idx]

In [ ]:
logisticRegressionclf.classes_

array([0, 1])

In [22]:
print("\nTop Machine-Generated Indicators:\n")
for f, w in top_machine_features:
    print(f"{f}: {w:.4f}")

print("\nTop Human-Written Indicators:\n")
for f, w in top_human_features:
    print(f"{f}: {w:.4f}")


Top Machine-Generated Indicators:

python: 81.9746
endoftext: 52.5180
cpp: 37.1156
explanation: 30.8145
python3: 25.3339
__name__: 24.5292
namespace: 21.4269
example: 20.9732
here: 20.3816
anything: 19.7137
usage: 19.0571
the: 17.5072
__init__: 16.9584
with: 16.8973
__main__: 15.3327
only: 15.1481
comments: 14.9836
standard: 14.3507
stdio: 13.9766
java: 13.5112
leetcode: 13.4920
raw_input: 13.1546
based: 12.8367
please: 12.4764
examples: 12.3218
code: 12.2041
xrange: 12.0287
without: 11.3749
constraints: 11.2903
approach: 11.1647

Top Human-Written Indicators:

class: -22.3508
provide: -19.7939
stdc: -18.8815
iowrapper: -15.1861
parsedouble: -11.5425
called: -11.1754
lang: -10.3271
him: -9.6813
nextdouble: -9.2193
signed: -9.2176
override: -9.1441
online_judge: -9.1196
util: -8.3616
become: -7.8642
ob: -7.2721
throw: -7.2391
fstat: -7.0350
writable: -6.4693
hasmoreelements: -6.4434
st_size: -6.4329
__starting_point: -5.9960
such: -5.9597
typename: -5.9407
except: -5.9332
running: -5.9

## Error Analysis


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Human", "Machine"],
    yticklabels=["Human", "Machine"],
)
plt.xlabel(" ")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
# Save figure as PDF
plt.savefig("plots/confusion_matrix_svm_human_vs_machine.pdf", bbox_inches="tight")
plt.show()

In [ ]:
test_data["label"].value_counts()

In [ ]:
report = classification_report(y_test, pred, output_dict=True)
pd.DataFrame(report)

In [ ]:
# Generate classification report
report = classification_report(y_test, pred, output_dict=True)
df = pd.DataFrame(report).transpose()

# Keep only class rows (drop accuracy, macro avg, weighted avg)
metrics = df.iloc[:-3][["precision", "recall", "f1-score"]]

# Rename classes
metrics.index = metrics.index.map({"0": "Human", "1": "Machine"})

# Convert to long format for seaborn
metrics_long = metrics.reset_index().melt(
    id_vars="index", var_name="Metric", value_name="Score"
)

# Plot with seaborn
plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_long, x="index", y="Score", hue="Metric")

plt.title("Classification Report Metrics")
plt.xlabel(" ")
plt.ylabel(" ")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("plots/classification_report_metrics.pdf", bbox_inches="tight")
plt.show()

### Interpretation

Class 0 interpretation

- When model predicts 0 → usually correct (high precision)

- But it rarely predicts 0 → so it misses many real 0s (low recall)

Class 1 interpretation

- Model predicts 1 a LOT → catches most class 1’s (high recall)

- But many of the "1" predictions are wrong (low precision)


## Wrong Predictions


In [ ]:
errors = []
for i in range(len(y_test)):
    if y_test[i] != pred[i]:
        errors.append({"text": X_test_text[i], "true": y_test[i], "pred": pred[i]})

import pandas as pd

errors_df = pd.DataFrame(errors)
errors_df.head(20)

## Look at the Worst Predictions (Highest Probability Errors)


In [ ]:
probs = logisticRegressionclf.predict_proba(X_test_tfidf)
confidences = probs.max(axis=1)

wrong_indices = pred != y_test

worst_df = pd.DataFrame(
    {"text": X_test_text, "true": y_test, "pred": pred, "confidence": confidences}
)[wrong_indices]

worst_df = worst_df.sort_values(by="confidence", ascending=False)
worst_df.head(15)

In [ ]:
from itertools import chain
from datasets import load_dataset
from tqdm import tqdm


def predict_with_svm(logistReg, vectorizer, parquet_path, output_path, batch_size=512):
    """
    Streaming inference for logistReg when parquet file has:
    ['code', 'generator', 'label', 'language']
    """

    # Stream parquet
    ds = load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)

    it = iter(ds)
    first = next(it)

    # Must contain 'code' column
    if not {"ID", "code"}.issubset(first.keys()):
        raise ValueError("Parquet file must contain 'ID' and 'code' columns")

    # Rechain for full iterator
    stream = chain([first], it)

    # Streaming batcher
    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    # Write CSV
    with open(output_path, "w") as f:
        f.write("ID,label\n")

        for batch in tqdm(batcher(stream, batch_size), desc="Predicting (SVM)"):
            print(batch)
            codes = [row["code"] for row in batch]
            ids = [row["ID"] for row in batch]

            # Vectorize
            X = vectorizer.transform(codes)

            # Predict

            probs = logistReg.predict_proba(X)
            preds = (probs[:, 1] >= 0.6).astype(int)

            # Write predictions with generated IDs
            for ex_id, pred in zip(ids, preds):
                f.write(f"{ex_id},{pred}\n")

    print(f"✅ Predictions saved to {output_path}")

In [ ]:
predict_with_svm(
    logistReg=logisticRegressionclf,
    vectorizer=tfidf,
    parquet_path="test.parquet",
    output_path="logisticRegression.csv",
    batch_size=512,
)